<a href="https://colab.research.google.com/github/Karandeep09/transformer/blob/main/Encoder_Decoder_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import torch
import torch.nn as nn
import math
class Attention(nn.Module):
  def __init__(self, d_model):
    super().__init__()
    self.k_w = nn.Linear(d_model, d_model, bias=False)
    self.q_w = nn.Linear(d_model, d_model, bias=False)
    self.v_w = nn.Linear(d_model, d_model, bias=False)
  def forward(self, embeddings_q, embeddings_k, embeddings_v, mask = None):
    k = self.k_w(embeddings_k)
    q = self.q_w(embeddings_q)
    v = self.v_w(embeddings_v)
    sims = q @ k.transpose(-2, -1)
    scaled_sims = sims / math.sqrt(k.size(-1))
    if mask is not None:
      scaled_sims = scaled_sims.masked_fill(mask, value = -1e9)
    attention_percents = torch.softmax(scaled_sims, dim=-1)
    return attention_percents @ v


In [9]:
embeddings_k = torch.tensor([[1.3,2.4],[0.5, 0.9],[1.3, 4.2]])
embeddings_q = torch.tensor([[1.3,2.4],[0.5, 0.9],[1.3, 4.2]])
embeddings_v = torch.tensor([[1.3,2.4],[0.5, 0.9],[1.3, 4.2]])
mask = torch.tril(torch.ones((3,3))) == 0
print(mask)
attention = Attention(2)
attention_scores = attention(embeddings_q, embeddings_k, embeddings_v, mask)
print(attention_scores)

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])
tensor([[0.7632, 0.6878],
        [0.5169, 0.4666],
        [0.6528, 0.5718]], grad_fn=<MmBackward0>)


In [10]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads = 1):
    super().__init__()
    self.num_heads = num_heads
    self.heads = nn.ModuleList([Attention(d_model) for _ in range(num_heads)])

  def forward(self, embeddings_q, embeddings_k, embeddings_v, mask = None):
    return torch.cat([head(embeddings_q, embeddings_k, embeddings_v, mask) for head in self.heads], dim = -1)




In [16]:
embeddings_k = torch.tensor([[1.3,2.4],[0.5, 0.9],[1.3, 4.2]])
embeddings_q = torch.tensor([[1.3,2.4],[0.5, 0.9],[1.3, 4.2]])
embeddings_v = torch.tensor([[1.3,2.4],[0.5, 0.9],[1.3, 4.2]])
multi_head_attention = MultiHeadAttention(d_model = 2, num_heads = 2)
attention_scores = multi_head_attention(embeddings_q, embeddings_k, embeddings_v)
print(attention_scores)

tensor([[-0.4003, -1.2990, -0.7859,  0.2067],
        [-0.5350, -1.6317, -1.0438,  0.2592],
        [-0.2994, -1.0440, -0.6875,  0.1852]], grad_fn=<CatBackward0>)
